# Lab 04: The Equilibrium-Point Hypothesis / λ Model

**Computational Sensorimotor Control — Week 4**

**Objectives:**
- Install and import the `smc` plant library from GitHub
- Replace direct activation with λ threshold control
- Implement constant-rate λ shifts for reaching movements
- Explore R-command (movement) and C-command (stiffness)
- Test equifinality — the spring-like signature of EPH
- Observe workspace dependence — the limitation that motivates Week 5

**Key equation:**
$$A = [l - \lambda + \mu \cdot \dot{l}]^+$$


---
## Part 1: Installing and Importing the Plant Library

This is the first lab where you **import** the arm model instead of building it. The `smc` library contains the same `Arm`, `Muscle`, and dynamics code you built in Weeks 1–3.


In [ ]:
# Install the smc library (only needs to run once per session)
!pip3 install git+https://github.com/tarkeshsingh/computational-sensorimotor-control.git#subdirectory=smc -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

from smc import (
    Arm, make_muscles, simulate_direct, simulate_lambda,
    Q_REF, L1, L2, MUSCLE_DEFS, Q1_LIMITS, Q2_LIMITS,
    mass_matrix, joint_accelerations, rk4_step, C_EXP, MU_LAMBDA,
)

arm = Arm()
start_hand = arm.forward_kinematics(Q_REF)
print(f"Reference posture: θ₁={np.rad2deg(Q_REF[0]):.0f}°, θ₂={np.rad2deg(Q_REF[1]):.0f}°")
print(f"Hand position: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm")
print(f"Muscles: {[m[0] for m in MUSCLE_DEFS]}")
print("✓ Plant loaded")

---
## Part 2: The HW03 Problem — Why We Need λ Control
Reproduce the HW03 finding: direct activation with B = 0 cannot settle.


In [ ]:
def triphasic_hw03(t):
    a = np.zeros(6)
    if 0.02 <= t < 0.20: a[0] = 1.0; a[2] = 0.4
    if 0.18 <= t < 0.30: a[3] = 0.8; a[5] = 0.3
    if t >= 0.28: a[0] = 0.25; a[3] = 0.25
    return a

t_A, _, h_A, _ = simulate_direct(triphasic_hw03, T=1.0, B=3.0)
t_B, _, h_B, _ = simulate_direct(triphasic_hw03, T=1.0, B=0.0)

d_A = np.linalg.norm(h_A - start_hand, axis=1) * 100
d_B = np.linalg.norm(h_B - start_hand, axis=1) * 100
v_B = np.zeros(len(t_B)); v_B[1:] = np.linalg.norm(np.diff(h_B,axis=0)/0.0001,axis=1)*100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax1.plot(t_A*1000, d_A, '#2E86AB', lw=2.5, label='Model A (B=3.0)')
ax1.plot(t_B*1000, d_B, '#E74C3C', lw=2.5, label='Model B (B=0)')
ax1.set_ylabel('Displacement (cm)'); ax1.legend()
ax1.set_title('HW03 Revisited: Direct Activation Cannot Settle Without Damping')
ax2.plot(t_B*1000, v_B, '#E74C3C', lw=2.5)
ax2.axhline(1.0, color='gray', ls=':', alpha=0.5)
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand Speed (cm/s)')
plt.tight_layout(); plt.show()
print(f"Model B final velocity: {v_B[-1]:.1f} cm/s — still moving.")

---
## Part 3: The λ Activation Law — Helper Functions

The threshold law: $A = [l - \lambda + \mu \cdot \dot{l}]^+$

We need two helper functions:
1. `lambda_for_posture(q, cc_mm)` — compute λ values for a given posture with cc_mm millimeters of co-contraction
2. `make_ramp(lam_init, lam_final, t_start, duration)` — create a linear λ ramp


In [ ]:
def lambda_for_posture(q_target, cc_mm=10.0):
    """Compute λ values that hold the arm at q_target with cc_mm of co-contraction."""
    muscles = make_muscles()
    lams = np.zeros(6)
    for j, m in enumerate(muscles):
        lams[j] = m.length(q_target) - cc_mm / 1000.0
    return lams

def make_ramp(lam_init, lam_final, t_start=0.05, duration=0.35):
    """Constant-rate λ ramp. Before t_start: hold. During: interpolate. After: hold."""
    def fn(t):
        if t < t_start: return lam_init.copy()
        elif t < t_start + duration:
            return lam_init + (t - t_start) / duration * (lam_final - lam_init)
        else: return lam_final.copy()
    return fn

def ik(x, y):
    """Inverse kinematics (elbow-up)."""
    return arm.inverse_kinematics(x, y)

CC_MM = 10.0  # co-contraction level (mm) — used throughout
print("Helper functions defined ✓")

---
## Part 4: Your First λ Reach
Reach 8 cm to the right using nothing but a constant-rate λ ramp.


In [ ]:
target_xy = np.array([start_hand[0] + 0.08, start_hand[1]])
target_q = ik(target_xy[0], target_xy[1])

lam_start = lambda_for_posture(Q_REF, CC_MM)
lam_target = lambda_for_posture(target_q, CC_MM)
delta_lam = lam_target - lam_start

print(f"Δλ range: {delta_lam.min()*1000:.1f} to {delta_lam.max()*1000:.1f} mm")
print("A few mm of threshold shift is the entire command.")

lam_fn = make_ramp(lam_start, lam_target)
t_lam, s_lam, h_lam, a_lam = simulate_lambda(lam_fn, T=1.2)

d_lam = np.linalg.norm(h_lam - start_hand, axis=1) * 100
v_lam = np.zeros(len(t_lam))
v_lam[1:] = np.linalg.norm(np.diff(h_lam, axis=0)/0.0001, axis=1) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax1.plot(t_lam*1000, d_lam, '#27AE60', lw=2.5); ax1.axhline(8.0, color='gray', ls=':')
ax1.axvspan(50, 400, alpha=0.05, color='#2E86AB')
ax1.set_ylabel('Displacement (cm)'); ax1.set_title('λ Control: Smooth Reaching')
ax2.plot(t_lam*1000, v_lam, '#27AE60', lw=2.5)
ax2.axvspan(50, 400, alpha=0.05, color='#2E86AB')
ax2.text(225, v_lam.max()*0.85, 'λ ramp', ha='center', color='#2E86AB', fontweight='bold')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand Speed (cm/s)')
plt.tight_layout(); plt.show()

print(f"Peak velocity: {v_lam.max():.1f} cm/s (bell-shaped ✓)")
print(f"Final velocity: {v_lam[-1]:.1f} cm/s")
print("No B, no G, no trajectory planning — just a λ ramp.")

---
## Part 5: λ vs Direct Activation — Same Plant, Different Result


In [ ]:
# TODO: COMPLETE THIS CELL
# Follow the instructions in the markdown above



---
## Part 6: Center-Out Reaching — 8 Directions


In [ ]:
# TODO: COMPLETE THIS CELL
# Follow the instructions in the markdown above



---
## Part 7: Threshold Displacement Traces

$A = [l - \lambda + \mu \cdot \dot{l}]^+$ in mm for each muscle during the 0° reach.

**These are threshold displacements** (input to the force curve), **not EMG or force**.


In [ ]:
li = lambda_for_posture(Q_REF, CC_MM)
lf = lambda_for_posture(ik(start_hand[0]+0.08, start_hand[1]), CC_MM)
t_r, _, _, a_r = simulate_lambda(make_ramp(li, lf), T=1.2)

mnames = ['Pec','BicL','BicS','Delt','TriL','TriLg']
mcols = ['#E74C3C','#E67E22','#E8735A','#2E86AB','#2E86AB','#27AE60']

fig, axes = plt.subplots(3, 2, figsize=(13, 8), sharex=True)
for mi in range(6):
    ax = axes[mi//2, mi%2]
    ax.plot(t_r*1000, a_r[:,mi]*1000, color=mcols[mi], lw=2)
    ax.fill_between(t_r*1000, a_r[:,mi]*1000, alpha=0.15, color=mcols[mi])
    ax.set_ylabel(f'{mnames[mi]} (mm)', fontsize=10, fontweight='bold', color=mcols[mi])
    ax.set_xlim(0, 800); ax.axvspan(50, 400, alpha=0.05, color='#2E86AB')
    if mi >= 4: ax.set_xlabel('Time (ms)')
fig.suptitle('Threshold Displacement A = [l − λ + μ·dl/dt]⁺ (mm)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()
print("Teal shading = λ ramp period (50–400 ms)")

---
## Part 7b: Calcium-Filtered Muscle Force — λ vs Direct Activation

The threshold displacement traces (Part 7) show *what drives* force production. Now let's look at the actual **calcium-filtered muscle force** — the output of the Week 2 calcium dynamics — for two opposing muscles.

**Your tasks:**
1. Complete `sim_forces_lambda()` — fill in the two lines that compute force and store the calcium state (hint: look at how `sim_forces_direct` does it above)
2. Call both simulation functions with the correct arguments
3. Compute the ratio of λ peak force to direct peak force

The plotting code is provided. This is Figure 10 from the lecture notes.


In [ ]:
# Helper: simulate and track per-muscle calcium-filtered forces
from smc import make_muscles as _mk_f, rk4_step as _rk4_f, joint_accelerations as _ja_f

def sim_forces_direct(act_fn, T=1.0, dt=0.0001, B=0.0):
    """Simulate with direct activation, returning per-muscle calcium-filtered forces."""
    muscles = _mk_f()
    state = np.concatenate([Q_REF, [0., 0.]])
    t = np.arange(0, T, dt); n = len(t)
    forces = np.zeros((n, 6))
    for i in range(n):
        q, qd = state[:2], state[2:]
        a = act_fn(t[i]); tau = np.zeros(2)
        for j, m in enumerate(muscles):
            f = m.compute_force_direct(a[j], q, qd, dt)
            forces[i, j] = m.ca[0]  # calcium-filtered force state (Week 2!)
            tau[0] += m.r_sh * f; tau[1] += m.r_el * f
        state = _rk4_f(state, lambda s: np.array([s[2],s[3],*_ja_f(s[:2],s[2:],tau,B)]), dt)
    return t, forces

def sim_forces_lambda(lam_fn, T=1.0, dt=0.0001):
    """Simulate with λ control, returning per-muscle calcium-filtered forces.
    
    TODO: COMPLETE the force computation inside the loop.
    Hint: use m.compute_force_lambda() instead of m.compute_force_direct()
    Remember: compute_force_lambda returns TWO values (force, activation)
    """
    muscles = _mk_f()
    state = np.concatenate([Q_REF, [0., 0.]])
    t = np.arange(0, T, dt); n = len(t)
    forces = np.zeros((n, 6))
    for i in range(n):
        q, qd = state[:2], state[2:]
        lams = lam_fn(t[i]); tau = np.zeros(2)
        for j, m in enumerate(muscles):
            # ──── TODO: COMPLETE THESE TWO LINES ────
            # 1. Compute force using λ control (hint: m.compute_force_lambda takes 4 args)
            # 2. Store the calcium-filtered force state in forces[i, j]
            f, _ = ...   # TODO: call m.compute_force_lambda(???, q, qd, dt)
            forces[i, j] = ...   # TODO: what attribute of m stores the filtered force? (same as direct version above)
            # ────────────────────────────────────────
            tau[0] += m.r_sh * f; tau[1] += m.r_el * f
        state = _rk4_f(state, lambda s: np.array([s[2],s[3],*_ja_f(s[:2],s[2:],tau,0.)]), dt)
    return t, forces

# ──── TODO: SIMULATE BOTH MODELS ────
# Direct activation: use triphasic_hw03 from Part 2, T=1.2, B=3.0
t_fd, f_direct = sim_forces_direct(...)   # TODO: fill in arguments

# λ control: use the same reach as Part 4
li = lambda_for_posture(Q_REF, CC_MM)
lf = lambda_for_posture(ik(start_hand[0]+0.08, start_hand[1]), CC_MM)
t_fl, f_lambda = sim_forces_lambda(...)   # TODO: fill in argument (what function does make_ramp return?)
# ────────────────────────────────────

# ──── Plotting (provided) ────
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax = axes[0]
ax.plot(t_fd*1000, f_direct[:, 0], '#E74C3C', lw=2.5, label='Direct activation (Model A, B=3.0)')
ax.plot(t_fl*1000, f_lambda[:, 0], '#27AE60', lw=2.5, label='λ control (B=0)')
ax.set_ylabel('Pectoralis\nForce (N)', fontsize=12, fontweight='bold')
ax.set_title('Shoulder Flexor (Pectoralis)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.2)
ax.axvspan(20, 200, alpha=0.05, color='#E74C3C')
ax.axvspan(50, 400, alpha=0.04, color='#27AE60')

ax = axes[1]
ax.plot(t_fd*1000, f_direct[:, 3], '#E74C3C', lw=2.5, label='Direct activation (Model A, B=3.0)')
ax.plot(t_fl*1000, f_lambda[:, 3], '#27AE60', lw=2.5, label='λ control (B=0)')
ax.set_ylabel('Deltoid\nForce (N)', fontsize=12, fontweight='bold')
ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_title('Shoulder Extensor (Deltoid)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.2)
ax.axvspan(180, 300, alpha=0.05, color='#E74C3C')
ax.axvspan(50, 400, alpha=0.04, color='#27AE60')

for ax in axes: ax.set_xlim(0, 1000)
fig.suptitle('Calcium-Filtered Muscle Force: Direct Activation vs. λ Control', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# ──── TODO: PRINT COMPARISON ────
# Print peak forces for both controllers (muscles 0 and 3)
# What is the ratio of λ peak force to direct peak force?
print(f"Direct: pec = {f_direct[:,0].max():.0f} N, delt = {f_direct[:,3].max():.0f} N")
print(f"λ:      pec = {f_lambda[:,0].max():.0f} N, delt = {f_lambda[:,3].max():.0f} N")
# TODO: compute and print the ratio
# print(f"λ/direct ratio (pec): {???:.1%}")

---
## Part 8: Equifinality — The Spring-Like Signature

Perturb the arm mid-reach. Does it return to the same endpoint?


In [ ]:
li = lambda_for_posture(Q_REF, CC_MM)
lf = lambda_for_posture(ik(start_hand[0]+0.08, start_hand[1]), CC_MM)
lam_fn = make_ramp(li, lf)

t_np, _, h_np, _ = simulate_lambda(lam_fn, T=1.5)
def pert1(t): return np.array([5.0, -3.0]) if 0.20<t<0.25 else np.zeros(2)
def pert2(t): return np.array([-3.0, 5.0]) if 0.30<t<0.35 else np.zeros(2)
_, _, h_p1, _ = simulate_lambda(lam_fn, T=1.5, perturb_fn=pert1)
_, _, h_p2, _ = simulate_lambda(lam_fn, T=1.5, perturb_fn=pert2)

dev1 = np.linalg.norm(h_p1 - h_np, axis=1) * 100
dev2 = np.linalg.norm(h_p2 - h_np, axis=1) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for h,col,lab in [(h_np,'#2E86AB','Unperturbed'),(h_p1,'#E74C3C','Pert. 200ms'),(h_p2,'#E67E22','Pert. 300ms')]:
    ax1.plot(h[:,0]*100,h[:,1]*100,color=col,lw=2.5,label=lab)
    ax1.plot(h[-1,0]*100,h[-1,1]*100,'o',color=col,ms=10,markeredgecolor='k',markeredgewidth=1.5,zorder=5)
ax1.plot(start_hand[0]*100,start_hand[1]*100,'ko',ms=12,zorder=5)
ax1.set_xlabel('X (cm)'); ax1.set_ylabel('Y (cm)'); ax1.set_title('Paths Converge')
ax1.legend(); ax1.set_aspect('equal')

ax2.plot(t_np*1000, dev1, '#E74C3C', lw=2.5, label=f'Pert 200ms (peak: {dev1.max():.1f} cm)')
ax2.plot(t_np*1000, dev2, '#E67E22', lw=2.5, label=f'Pert 300ms (peak: {dev2.max():.1f} cm)')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Deviation (cm)')
ax2.set_title('Path Deviation → Converges to < 0.01 cm'); ax2.legend(); ax2.set_xlim(0,1200)

plt.tight_layout(); plt.show()
print(f"Peak deviation: {max(dev1.max(), dev2.max()):.1f} cm")
print(f"Final deviation: {max(dev1[-1], dev2[-1]):.3f} cm — equifinality ✓")

---
## Part 9: Workspace Dependence

Same Δλ, different arm configuration → different hand paths. This is the cost of controlling in muscle space.


In [ ]:
target_q_A = ik(start_hand[0]+0.08, start_hand[1])
li_A = lambda_for_posture(Q_REF, CC_MM)
lf_A = lambda_for_posture(target_q_A, CC_MM)
delta_lam = lf_A - li_A

q_startB = np.array([np.radians(90), np.radians(45)])
hand_B = arm.forward_kinematics(q_startB)
li_B = lambda_for_posture(q_startB, CC_MM)
lf_B = li_B + delta_lam  # same Δλ!

# Pre-equilibrate reach B (passive springs not balanced at q_startB)
from smc import make_muscles as _mk, rk4_step as _rk4, joint_accelerations as _ja
def sim_eq(q0, lam_fn, T=1.2, dt=0.0001, T_eq=0.5):
    muscles = _mk(); state = np.concatenate([q0,[0.,0.]]); lam0 = lam_fn(0)
    for _ in range(int(T_eq/dt)):
        q,qd=state[:2],state[2:]; tau=np.zeros(2)
        for j,m in enumerate(muscles):
            f,_=m.compute_force_lambda(lam0[j],q,qd,dt); tau[0]+=m.r_sh*f; tau[1]+=m.r_el*f
        state=_rk4(state,lambda s:np.array([s[2],s[3],*_ja(s[:2],s[2:],tau,0.)]),dt)
    t=np.arange(0,T,dt); hand=np.zeros((len(t),2))
    for i in range(len(t)):
        q,qd=state[:2],state[2:]; hand[i]=arm.forward_kinematics(q); lams=lam_fn(t[i]); tau=np.zeros(2)
        for j,m in enumerate(muscles):
            f,_=m.compute_force_lambda(lams[j],q,qd,dt); tau[0]+=m.r_sh*f; tau[1]+=m.r_el*f
        state=_rk4(state,lambda s:np.array([s[2],s[3],*_ja(s[:2],s[2:],tau,0.)]),dt)
    return t, hand

t_A2, h_A2 = sim_eq(Q_REF, make_ramp(li_A, lf_A), T_eq=0.3)
t_B2, h_B2 = sim_eq(q_startB, make_ramp(li_B, lf_B), T_eq=0.5)
v_A2=np.zeros(len(t_A2)); v_A2[1:]=np.linalg.norm(np.diff(h_A2,axis=0)/0.0001,axis=1)*100
v_B2=np.zeros(len(t_B2)); v_B2[1:]=np.linalg.norm(np.diff(h_B2,axis=0)/0.0001,axis=1)*100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for q_s,col in [(Q_REF,'#2E86AB'),(q_startB,'#E74C3C')]:
    ex,ey=L1*np.cos(q_s[0]),L1*np.sin(q_s[0]); hx,hy=arm.forward_kinematics(q_s)
    ax1.plot([0,ex*100],[0,ey*100],color=col,lw=4,alpha=0.25)
    ax1.plot([ex*100,hx*100],[ey*100,hy*100],color=col,lw=4,alpha=0.25)
ax1.plot(h_A2[:,0]*100,h_A2[:,1]*100,'#2E86AB',lw=2.5,label='Reach A (55°,75°)')
ax1.plot(h_B2[:,0]*100,h_B2[:,1]*100,'#E74C3C',lw=2.5,label='Reach B (90°,45°)')
ax1.set_xlabel('X (cm)'); ax1.set_ylabel('Y (cm)'); ax1.set_title('Same Δλ, Different Configs')
ax1.legend(); ax1.set_aspect('equal')
ax2.plot(t_A2*1000,v_A2,'#2E86AB',lw=2.5,label='Reach A'); ax2.plot(t_B2*1000,v_B2,'#E74C3C',lw=2.5,label='Reach B')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Speed (cm/s)'); ax2.set_title('Hand Speed')
ax2.legend(); ax2.set_xlim(0,800)
plt.tight_layout(); plt.show()
print("Same Δλ → different paths. Workspace dependence. Week 5 addresses this.")

---
## Summary

1. **λ control settles without external damping** — the HW03 Model B problem is solved
2. **The control signal is simple** — constant-rate λ ramps of a few mm
3. **Equifinality holds** — perturbations don't change the endpoint
4. **Workspace dependence** — same Δλ ≠ same hand path across configurations

**Next week:** VITE and Minimum-Jerk — planning in task space.
